In [0]:
"""Figures for the progress report. All numbers are copied from notebook
outputs (NB05b, NB10, NB11, NB12, NB14, NB15b, NB16, NB17, NB18) as reported
in the 2026-10-14 session; each figure's source is stated in its caption."""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

OUT = "figures/"
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a",
     "yellow": "#eda100", "magenta": "#e87ba4"}
INK, INK2, GRID, NEUTRAL = "#0b0b0b", "#52514e", "#e6e5e0", "#a3a29d"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8.5, "axes.labelsize": 8.5,
    "axes.titlesize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
    "ytick.color": INK2, "text.color": INK, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False,
    "savefig.bbox": "tight", "pdf.fonttype": 42,
})

def save(fig, name):
    fig.savefig(OUT + name + ".pdf")
    fig.savefig(OUT + name + ".png", dpi=200)
    plt.close(fig)

# ---------------------------------------------------------------------------
# F1 — coverage of experimental variables (NB05b)
# ---------------------------------------------------------------------------
fill = {"catalyst_mass_g": 99.0, "lamp_power_W": 92.9, "thermal_T_C": 90.3,
        "cutoff_nm_final": 86.0, "thermal_time_h": 84.7, "volume_mL": 83.1,
        "catalyst_conc_g_L": 83.1, "lamp_power_per_mL": 78.7,
        "co_catalyst_loading": 58.3, "particle_size_nm": 53.4, "ssa_m2_g": 49.9,
        "reaction_time_h": 48.2, "prep_multistep": 39.1, "T_reaction_C": 27.5,
        "organic_vol_fraction": 27.5, "irradiance_mW_cm2": 13.0, "pH": 9.6}
names = list(fill)[::-1]
vals = [fill[n] for n in names]
fig, ax = plt.subplots(figsize=(5.6, 3.9))
cols = [C["blue"] if v >= 20 else NEUTRAL for v in vals]
ax.barh(names, vals, color=cols, height=0.62)
for i, v in enumerate(vals):
    ax.text(v + 1, i, f"{v:.0f}%", va="center", fontsize=7.5, color=INK2)
ax.axvline(20, color=INK2, lw=0.8, ls="--")
ax.text(21, len(names) - 0.4, "corte de 20%", fontsize=7.5, color=INK2)
ax.set_xlim(0, 108)
ax.set_xlabel("Linhas com valor reportado (%, n = 691)")
ax.grid(axis="y", visible=False)
save(fig, "f01_coverage")

# ---------------------------------------------------------------------------
# F2 — feature layers, single 5-fold GroupKFold (NB10 v1 / v2)
# ---------------------------------------------------------------------------
models = ["RF", "XGBoost", "CatBoost"]
L1 = [-0.162, -0.298, -0.282]
L3v1 = [-0.056, -0.005, -0.028]
L3v2 = [0.072, 0.153, 0.106]
x = np.arange(3); w = 0.26
fig, ax = plt.subplots(figsize=(5.2, 3.0))
for k, (lab, v, c) in enumerate([("L1: material", L1, C["blue"]),
                                 ("L3: + experimental (X_exp)", L3v1, C["orange"]),
                                 ("L3: + experimental (X_exp_v2)", L3v2, C["aqua"])]):
    b = ax.bar(x + (k - 1) * w, v, w * 0.92, color=c, label=lab)
    for xi, vi in zip(x + (k - 1) * w, v):
        ax.text(xi, vi + (0.012 if vi >= 0 else -0.012), f"{vi:+.2f}", ha="center",
                va="bottom" if vi >= 0 else "top", fontsize=7, color=INK2)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylabel("R² (log), GroupKFold por DOI (5 folds)")
ax.set_ylim(-0.36, 0.22)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, fontsize=7)
ax.grid(axis="x", visible=False)
save(fig, "f02_layers")

# ---------------------------------------------------------------------------
# F3 — ablation (NB11) and composition contribution (NB12 + NB18), paired ΔR²
# ---------------------------------------------------------------------------
abl = [("EXP_ONLY (sem material)", -0.001, -0.244, 0.243),
       ("L3 − solução/reação", 0.011, -0.092, 0.126),
       ("L3 − material medido", -0.011, -0.155, 0.094),
       ("L3 − normalização por massa", -0.025, -0.177, 0.117),
       ("L3 − cocatalisador", -0.060, -0.174, 0.043),
       ("L3 − síntese", -0.064, -0.186, 0.168),
       ("L3 − reator/luz", -0.157, -0.416, 0.105),
       ("L1 (só material)", -0.472, -0.912, -0.126)]
comp = [("+ frações de ânion (V2)", 0.018, -0.060, 0.127),
        ("+ X_material antigo (V1)", 0.002, -0.203, 0.249),
        ("+ X_material_v2 (com ânions)", -0.001, -0.189, 0.290),
        ("+ compacto sem Eg", -0.010, -0.231, 0.181),
        ("+ compacto (16, com Eg medido)", -0.047, -0.283, 0.196),
        ("só X_material_v2", -0.336, -0.695, 0.167),
        ("só compacto", -0.509, -1.024, 0.153)]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), gridspec_kw={"wspace": 1.35})
for ax, data, title, ref in [(axes[0], abl, "Ablação (vs L3 completo)", "L3"),
                             (axes[1], comp, "Composição (vs EXP_ONLY)", "EXP")]:
    data = data[::-1]
    y = np.arange(len(data))
    for yi, (lab, m, lo, hi) in zip(y, data):
        ax.plot([lo, hi], [yi, yi], color=NEUTRAL, lw=1.6, solid_capstyle="round")
        ax.plot(m, yi, "o", ms=5.5, color=C["blue"], mec="white", mew=0.8)
    ax.axvline(0, color=INK2, lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels([d[0] for d in data])
    ax.set_title(title, loc="left")
    ax.set_xlabel("ΔR² pareado (log)")
    ax.grid(axis="y", visible=False)
save(fig, "f03_ablation_composition")

# ---------------------------------------------------------------------------
# F4 — within-paper pair accuracy by pair type (NB14 + NB18)
# ---------------------------------------------------------------------------
types = ["todos", "puro vs\nmodificado", "não-mistos", "mesma\ncomplexidade"]
series = [
    ("XGBoost, descritores sem ânions (NB14)", [0.643, 0.789, 0.605, 0.542], C["blue"], "o"),
    ("XGBoost, descritores com ânions (NB18)", [0.613, 0.768, 0.572, 0.503], C["orange"], "s"),
    ("Regra: modificado ganha", [np.nan, 0.786, np.nan, np.nan], C["aqua"], "D"),
    ("Regra: mais elementos ganha", [0.783, 0.837, 0.742, np.nan], C["yellow"], "^"),
    ("Regra: menor Eg ganha", [0.606, 0.729, 0.556, 0.481], C["magenta"], "v"),
]
fig, ax = plt.subplots(figsize=(6.0, 3.2))
ax.axhspan(0.5, 0.59, color="#f1f0ec", zorder=0)
ax.text(3.42, 0.545, "faixa nula\n(p95 ≈ 0.57–0.60)", fontsize=6.8, color=INK2, va="center")
ax.axhline(0.5, color=INK2, lw=0.8, ls="--")
off = np.linspace(-0.24, 0.24, len(series))
for (lab, v, c, mk), o in zip(series, off):
    ax.plot(np.arange(4) + o, v, mk, color=c, ms=6, mec="white", mew=0.7, label=lab,
            linestyle="none")
ax.set_xticks(range(4)); ax.set_xticklabels(types)
ax.set_xlim(-0.5, 4.1)
ax.set_ylim(0.44, 0.88)
ax.set_ylabel("Acurácia de pares (nulo = 0,5)")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax.grid(axis="x", visible=False)
save(fig, "f04_pairs")

# ---------------------------------------------------------------------------
# F5 — synthetic validation of the bias test (NB14 mock) vs real data (NB18)
# ---------------------------------------------------------------------------
groups = ["Sintético:\nsó viés", "Sintético:\nsó física", "Dados reais\n(NB18)"]
allp = [0.628, 0.752, 0.613]
nonm = [0.502, 0.746, 0.572]
x = np.arange(3); w = 0.34
fig, ax = plt.subplots(figsize=(4.6, 2.9))
for k, (lab, v, c) in enumerate([("todos os pares", allp, C["blue"]),
                                 ("pares não-mistos", nonm, C["orange"])]):
    ax.bar(x + (k - 0.5) * w, v, w * 0.9, color=c, label=lab)
    for xi, vi in zip(x + (k - 0.5) * w, v):
        ax.text(xi, vi + 0.006, f"{vi:.2f}", ha="center", fontsize=7, color=INK2)
ax.axhline(0.5, color=INK2, lw=0.8, ls="--")
ax.set_ylim(0.4, 0.82)
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylabel("Acurácia de pares (XGBoost OOF)")
ax.legend(fontsize=7.5, loc="upper left")
ax.grid(axis="x", visible=False)
save(fig, "f05_synthetic")

# ---------------------------------------------------------------------------
# F6 — band gap: featurisation and external data (NB17)
# ---------------------------------------------------------------------------
bg = [("Com ânions (RF)", 0.457, 0.308, 0.602, C["blue"]),
      ("Com ânions (XGBoost)", 0.414, 0.187, 0.565, C["blue"]),
      ("X_material antigo (RF)", 0.220, -0.008, 0.398, NEUTRAL),
      ("Mesmo featurizador, sem ânions (RF)", 0.130, -0.332, 0.319, NEUTRAL),
      ("+ externo como feature (Eg>0, RF)", 0.480, 0.333, 0.616, C["orange"]),
      ("+ externo, treino combinado (O/N)", 0.408, 0.157, 0.577, C["orange"]),
      ("Externo calibrado (O/N)", 0.182, 0.031, 0.265, C["orange"]),
      ("Externo zero-shot (O/N)", -0.119, -0.551, 0.168, C["orange"])][::-1]
fig, ax = plt.subplots(figsize=(5.8, 3.3))
ax.axvspan(0.90, 0.95, color="#f1f0ec", zorder=0)
ax.text(0.925, len(bg) - 0.4, "teto de\nmedição", fontsize=7, color=INK2, ha="center", va="top")
for yi, (lab, m, lo, hi, c) in enumerate(bg):
    ax.plot([lo, hi], [yi, yi], color=c, lw=1.6, alpha=0.55, solid_capstyle="round")
    ax.plot(m, yi, "o", ms=6, color=c, mec="white", mew=0.8)
    ax.text(hi + 0.02, yi, f"{m:.2f}", va="center", fontsize=7, color=INK2)
ax.axvline(0, color=INK2, lw=0.8)
ax.set_yticks(range(len(bg))); ax.set_yticklabels([b[0] for b in bg])
ax.set_xlim(-0.65, 1.0)
ax.set_xlabel("R² do band gap (CV por DOI + expurgo de composições; IC 95% bootstrap por DOI)")
ax.grid(axis="y", visible=False)
save(fig, "f06_bandgap")

# ---------------------------------------------------------------------------
# F7 — Eg reproducibility across papers (NB15b) and vs Matbench (NB16)
# ---------------------------------------------------------------------------
rep = [("SrTiO3", 15, 0.285), ("BaTiO3", 6, 0.139), ("LaFeO3", 6, 0.216),
       ("CaTiO3", 5, 0.077), ("La2Ti2O7", 4, 0.322), ("NaTaO3", 4, 0.208),
       ("LaMnO3", 4, 0.615), ("LaCoO3", 4, 0.593), ("KNbO3", 4, 0.073),
       ("La2NiO4", 3, 0.115), ("PbTiO3", 3, 0.100), ("Sr2TiO4", 3, 0.407)]
mb = [("BaIn0.5Ta0.5O3", 4.08, 4.17), ("BaTaO2N", 1.80, 1.78), ("BaSnO3", 3.01, 3.10),
      ("CaTaO2N", 2.50, 2.40), ("CaSnO3", 4.30, 4.40), ("La2CuO4", 1.80, 0.00),
      ("LaTaON2", 1.98, 2.00), ("LiTaO3", 5.12, 4.10), ("SrTaO2N", 2.07, 1.99),
      ("PrTaON2", 2.03, 2.00), ("Sr2TiO4", 2.95, 3.41), ("Sr3Ti2O7", 3.31, 3.26),
      ("SrSnO3", 4.10, 4.10)]
fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.1), gridspec_kw={"wspace": 0.55})
ax = axes[0]
rep_s = sorted(rep, key=lambda r: r[2])
ax.barh([r[0] for r in rep_s], [r[2] for r in rep_s], color=C["blue"], height=0.6)
for i, r in enumerate(rep_s):
    ax.text(0.72, i, f"n={r[1]}", va="center", ha="right", fontsize=6.8, color=INK2)
ax.axvline(0.139, color=INK2, lw=0.8, ls="--")
ax.text(0.145, len(rep_s) - 0.35, "mediana (25 comp.)\n0.14 eV", fontsize=6.8, color=INK2, va="bottom")
ax.set_ylim(-0.6, len(rep_s) + 0.9)
ax.set_xlabel("DP do Eg entre artigos (eV)")
ax.set_title("(a) Mesma composição, artigos diferentes", loc="left", fontsize=8.5)
ax.set_xlim(0, 0.75); ax.grid(axis="y", visible=False)
ax = axes[1]
p = np.array([m[1] for m in mb]); e = np.array([m[2] for m in mb])
ax.plot([0, 5.5], [0, 5.5], color=INK2, lw=0.8)
ax.plot(p, e, "o", color=C["orange"], ms=5.5, mec="white", mew=0.7)
OFFS = {"La2CuO4": (8, 2), "LiTaO3": (-14, -12), "Sr2TiO4": (-40, 2)}
for lab, a, b in mb:
    if lab in OFFS:
        ax.annotate(lab, (a, b), xytext=OFFS[lab], textcoords="offset points",
                    fontsize=6.8, color=INK2)
ax.set_xlim(0, 5.5); ax.set_ylim(-0.2, 5.5)
ax.set_xlabel("Eg no projeto (eV)"); ax.set_ylabel("Eg no Matbench (eV)")
ax.set_title("(b) Projeto vs compilação externa (13)", loc="left", fontsize=8.5)
save(fig, "f07_reproducibility")
print("figures written")

# ---------------------------------------------------------------------------
# F0 — v1 (previous report) vs v2: same targets, different validation/descriptors
# ---------------------------------------------------------------------------
eg = [("v1: hold-out por composto (GB)", 0.70, None, None, NEUTRAL),
      ("v2: CV por composição, X_material", 0.43, 0.15, 0.55, NEUTRAL),
      ("v2: CV por DOI, X_material", 0.19, 0.00, 0.44, NEUTRAL),
      ("v2: CV por DOI + expurgo, X_material", 0.14, -0.05, 0.38, NEUTRAL),
      ("v2: CV por DOI + expurgo, com ânions", 0.46, 0.31, 0.60, C["blue"])][::-1]
h2 = [("v1: CV por composto, material (RF)", 0.022, None, None, NEUTRAL),
      ("v1: CV por composto, material+thermo (RF)", 0.042, None, None, NEUTRAL),
      ("v1: teste hold-out por composto (RF)", 0.049, None, None, NEUTRAL),
      ("v2: CV por DOI, só material (XGB)", -0.403, -0.835, -0.080, C["orange"]),
      ("v2: CV por DOI, só condições (XGB)", 0.068, -0.338, 0.402, C["orange"]),
      ("v2: CV por DOI, condições+material (XGB)", 0.069, -0.284, 0.360, C["orange"])][::-1]
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.0), gridspec_kw={"wspace": 1.25})
for ax, data, title, xl in [(axes[0], eg, "(a) Band gap", (-0.2, 0.85)),
                            (axes[1], h2, "(b) Taxa de H$_2$ (log)", (-1.1, 0.5))]:
    for yi, (lab, m, lo, hi, c) in enumerate(data):
        if lo is not None:
            ax.plot([lo, hi], [yi, yi], color=c, lw=1.5, alpha=0.55, solid_capstyle="round")
        ax.plot(m, yi, "o" if "v2" in lab else "s", ms=5.5, color=c, mec="white", mew=0.7)
        if hi is not None and hi < 0:
            ax.text(lo - 0.03, yi, f"{m:.2f}", va="center", ha="right", fontsize=6.8, color=INK2)
        else:
            ax.text((hi if hi is not None else m) + 0.03, yi, f"{m:.2f}", va="center",
                    fontsize=6.8, color=INK2)
    ax.axvline(0, color=INK2, lw=0.8)
    ax.set_yticks(range(len(data))); ax.set_yticklabels([d[0] for d in data], fontsize=7.2)
    ax.set_xlim(*xl); ax.set_title(title, loc="left"); ax.set_xlabel("R²")
    ax.grid(axis="y", visible=False)
save(fig, "f00_v1_v2")
